# Лекция 2 · Python-проект
## От понятного кода к воспроизводимому запуску

**Информационные технологии при проектировании РиСО**  
**90 минут · Основы Python → структура проекта → окружение → проверка**

> Уметь получить код от LLM недостаточно. Нужно понять, **что** он вычисляет,
> **где** запускается и **как** убедиться в правильности результата.

Сквозной пример — **тележка, движущаяся вдоль прямой с постоянным ускорением**.
Математика намеренно простая: внимание уделяем коду и организации работы.

**После занятия:** открыть небольшой проект, найти вычислительную модель,
запустить ее в правильном окружении, проверить результат и поставить LLM конкретную задачу.

Актуализация материалов: **14 сентября 2026 года**.

<a id="navigation"></a>
## Как пользоваться блокнотом

Это конспект для показа преподавателем и повторения студентом. **Основной маршрут заканчивается
разделом 8**; приложения — справочник, а не дополнительная программа на те же 90 минут.

| Время | Блок | Что показываем |
|:--|:--|:--|
| 0–7 мин | [1. Notebook, Markdown, kernel](#notebook) | Где находится код и кто его выполняет |
| 7–30 мин | [2. Минимум Python](#python) | Типы, коллекции, условия, цикл |
| 30–40 мин | [3. Функция и проверка](#functions) | Контракт модели, ошибки, контрольные числа |
| 40–55 мин | [4. Из notebook в проект](#project) | Модуль, пакет, запуск, тесты |
| 55–63 мин | [5. venv и pip](#venv) | Какой Python? Куда установлен пакет? |
| 63–78 мин | [6. uv и Jupyter](#uv) | pyproject → lock → окружение → kernel |
| 78–85 мин | [7. Dev Container](#container) | Что добавляется сверх Python-пакетов |
| 85–90 мин | [8. Изменение с LLM](#llm) | Проверяемая постановка и выходной контроль |

**Ячейка Code** — исполняемый Python. **Команды терминала** даны только в текстовых блоках
и не выполняются кнопкой Run All. Установку программ выполняем **до занятия**, а не всей аудиторией во время объяснения.

Все исполняемые ячейки основного notebook используют стандартную библиотеку Python;
для их выполнения нужен уже работающий Jupyter kernel. Интернет, `uv`, Docker и доступ к LLM
для Run All не требуются. Раздел 4 создает только **новую временную папку** с демонстрационными файлами;
существующие проекты и окружение ядра не изменяются.

[Подготовка рабочего места](#setup) · [Шпаргалка Python](#cheatsheet) ·
[Практика после занятия](#practice) · [Источники и переработка старого курса](#sources)

<a id="notebook"></a>
# 1. Notebook — документ, kernel — исполнитель

В `.ipynb` находятся ячейки кода, текст, формулы и сохраненные результаты.
**Python выполняется в отдельном процессе — ядре (kernel)**. Вывод в открытом файле
мог быть сохранен вчера: наличие красивой таблицы не доказывает, что код запускается сейчас.

**Три разных места:** редактор показывает файл; терминал исполняет команды оболочки;
kernel исполняет Python-ячейки. Терминал и kernel могут использовать **разные интерпретаторы**. [S1, S2]

```text
.ipynb в редакторе  ── ячейка Python ──→  выбранный kernel
терминал           ── uv run ...    ──→  окружение проекта
```

**Проверка сдаваемой работы:** Restart Kernel → Run All → отсутствие неожиданных ошибок.

In [1]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Интерпретатор ядра:", sys.executable)
print("Рабочая папка:", Path.cwd())
print("Виртуальное окружение:", sys.prefix != sys.base_prefix)

Python: 3.12.2
Интерпретатор ядра: /home/mikhail/miniconda3/bin/python
Рабочая папка: /home/mikhail/c/Yandex/GD/00_teaching_2027/ИТ/lecture02_project/notebooks
Виртуальное окружение: False
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Markdown и LaTeX: минимальный язык инженерного отчета

Двойной щелчок по этой ячейке покажет исходный текст. `Shift+Enter` отобразит результат.

```markdown
## Условия расчета
**Предположение:** ускорение постоянно.
Функция `position` возвращает координату в метрах.

$$x(t) = x_0 + v_0 t + \frac{a t^2}{2}.$$
```

Результат записи формулы:

$$x(t) = x_0 + v_0 t + \frac{a t^2}{2}.$$

Здесь $x_0=1\,\text{м}$, $v_0=2\,\text{м/с}$, $a=0.5\,\text{м/с}^2$,
$t\geq0$. Мы считаем координату, **не длину пройденного пути**.

**Попробуйте:** измените заголовок и подпишите единицы измерения. Подробное оформление отчетов
будет развиваться дальше; сегодня достаточно заголовка, формулы и пояснения.

## Скрытое состояние: файл один, порядок запуска бывает разным

Выполните следующие две ячейки по порядку. Затем замените `3` на `10` в первой ячейке,
**не выполняйте ее**, а запустите только вторую. Результат не изменится:
редактирование текста не изменяет память kernel.

После опыта верните исходный код и выполните обе ячейки. Это переработанный пример
из старого введения; здесь нет незавершенных выражений, мешающих обычному Run All.

In [7]:
example_value = 3

In [6]:
print("Значение в памяти kernel:", example_value)

Значение в памяти kernel: 10


<a id="python"></a>
# 2. Python: минимум для чтения и изменения программы

Наша цель — **прочитать смысл**, изменить параметр и объяснить результат.
Не нужно за одно занятие выучить весь синтаксис.

Имя переменной связывается с объектом. Тип принадлежит объекту; имя можно связать с другим объектом.
`=` — присваивание, `==` — сравнение. Десятичный разделитель в коде — **точка**.

| Значение | Тип | Для чего в расчете |
|:--|:--|:--|
| `5` | `int` | Число измерений |
| `0.5` | `float` | Приближенное число с плавающей точкой |
| `True` / `False` | `bool` | Условие выполнено или нет |
| `"тележка"` | `str` | Название опыта |
| `None` | `NoneType` | Специальное значение «пока не задано» |

`None` — **тоже объект и значение**, не ноль, не пустая строка и не отсутствие переменной.

In [8]:
experiment_name = "Тележка — опыт 1"
sample_count = 5
acceleration_m_s2 = 0.5
is_ready = True
measured_position_m = None

print(experiment_name)
print(type(sample_count).__name__, type(acceleration_m_s2).__name__)
print(type(is_ready).__name__, type(measured_position_m).__name__)
print("Измерение отсутствует:", measured_position_m is None)

Тележка — опыт 1
int float
bool NoneType
Измерение отсутствует: True


## Выражение → вычисление → результат

`+`, `-`, `*`, `/`, `**` — сложение, вычитание, умножение, деление и степень.
`^` в Python **не возведение в степень**. Скобки лучше использовать явно, когда это улучшает чтение.

Имена с единицами измерения не заставляют Python проверять размерность, но помогают человеку
поймать ошибку. `print` выводит результат; f-строка вставляет значения в текст.

In [9]:
x0_m = 1.0
v0_m_s = 2.0
a_m_s2 = 0.5
t_s = 4.0

x_m = x0_m + v0_m_s * t_s + 0.5 * a_m_s2 * t_s**2
print(f"Через {t_s:.1f} с координата равна {x_m:.2f} м")
# Ручная проверка: 1 + 2*4 + 0.5*0.5*16 = 13.

Через 4.0 с координата равна 13.00 м


## Старый пример, который обязательно оставляем: `0.1 + 0.2`

`float` представляет большинство десятичных дробей приближенно. Поэтому точное сравнение
двух результатов вычислений иногда дает неожиданный ответ. Округление **отображения**
не повышает точность исходного значения. [S3]

Для проверок используем допуск, выбранный по смыслу задачи; около нуля особенно важен
абсолютный допуск. `1e-12` ниже — учебный допуск для этих маленьких чисел, не универсальное правило.

In [10]:
from math import isclose

print("0.1 + 0.2 =", 0.1 + 0.2)
print("Точное сравнение:", 0.1 + 0.2 == 0.3)
print("С допуском:", isclose(0.1 + 0.2, 0.3, rel_tol=1e-12, abs_tol=1e-12))

0.1 + 0.2 = 0.30000000000000004
Точное сравнение: False
С допуском: True


## Коллекции: список, кортеж, словарь

| Тип | Пример | Как получить элемент | Изменяемость |
|:--|:--|:--|:--|
| `list` — список | `[0.0, 1.0, 2.0]` | По индексу: `times[0]` | Можно менять элементы |
| `tuple` — кортеж | `(t_s, x_m)` | По индексу или распаковкой | Нельзя заменить элемент кортежа |
| `dict` — словарь | `{"x0_m": 1.0}` | По ключу: `params["x0_m"]` | Можно менять значения |

Вместо разрозненных значений получаем **серию измерений**, **одну точку** и **набор параметров**.
У списка первый индекс — `0`, последний — `-1`. В срезе `[a:b]` правая граница не включается.

In [11]:
times_s = [0.0, 1.0, 2.0, 3.0, 4.0]
print("Первый:", times_s[0], "Последний:", times_s[-1])
print("Срез [1:4]:", times_s[1:4])
print("Число точек:", len(times_s))

point = (4.0, 13.0)
point_time_s, point_x_m = point
print(f"Точка: t={point_time_s} с, x={point_x_m} м")

Первый: 0.0 Последний: 4.0
Срез [1:4]: [1.0, 2.0, 3.0]
Число точек: 5
Точка: t=4.0 с, x=13.0 м


In [12]:
params = {"x0_m": 1.0, "v0_m_s": 2.0, "a_m_s2": 0.5}
print("Начальная скорость:", params["v0_m_s"])
print("Все параметры:", params)

# Для отдельного опыта создаем копию словаря, а не меняем исходный.
other_params = params.copy()
other_params["a_m_s2"] = 0.0
print("Исходное ускорение:", params["a_m_s2"])
print("Ускорение во втором опыте:", other_params["a_m_s2"])

Начальная скорость: 2.0
Все параметры: {'x0_m': 1.0, 'v0_m_s': 2.0, 'a_m_s2': 0.5}
Исходное ускорение: 0.5
Ускорение во втором опыте: 0.0


## Условие и цикл

После `if`, `else`, `for` и `def` ставится **двоеточие**; вложенный блок выделяется
одинаковыми отступами. В наших примерах — четыре пробела.

`if` выбирает действие. `for` перебирает элементы коллекции. Не нужен счетчик индексов,
если нужны сами значения. `append` добавляет элемент в список.

In [9]:
time_to_check_s = 4.0

if time_to_check_s < 0:
    print("Для нашего опыта отрицательное время недопустимо")
else:
    print("Можно выполнять расчет")

positions_m = []
for time_s in times_s:
    coordinate_m = params["x0_m"] + params["v0_m_s"] * time_s + 0.5 * params["a_m_s2"] * time_s**2
    positions_m.append(coordinate_m)

print("Координаты:", positions_m)

Можно выполнять расчет
Координаты: [1.0, 3.25, 6.0, 9.25, 13.0]


### Проверка понимания · 1 минута

Что изменится, если поставить `positions_m = []` **внутрь** цикла?
Чему равно `times_s[1:3]`? Что означает `params["a_m_s2"]`?

<details>
<summary>Ответы — открыть после обсуждения</summary>

Список будет заново создаваться на каждом шаге; после цикла останется только последний результат.
Срез равен `[1.0, 2.0]`. По ключу `"a_m_s2"` извлекается ускорение; это не позиционный индекс.

</details>

<a id="functions"></a>
# 3. Функция — именованная модель с контрактом

Вынесем формулу в функцию. **Аргументы** задают условия, **`return`** возвращает результат.
`print` и `return` — разные действия: печать не заменяет возвращаемое число.

**Контракт:** аргументы — конечные числовые значения в СИ; `t_s >= 0`; результат — координата
в метрах. Ускорение может быть отрицательным. Проверяем отрицательное время явно;
полную валидацию всех возможных Python-объектов здесь не строим.

Аннотации `float` и `-> float` помогают читать интерфейс. Они **не проверяют типы автоматически
при запуске**; для этого нужны отдельные проверки или инструменты. Docstring описывает смысл. [S4]

In [10]:
def position(
    t_s: float,
    x0_m: float = 1.0,
    v0_m_s: float = 2.0,
    a_m_s2: float = 0.5,
) -> float:
    """Вернуть координату в метрах при t_s >= 0. Все аргументы — конечные числа."""
    if t_s < 0:
        raise ValueError("t_s must be non-negative")
    return x0_m + v0_m_s * t_s + 0.5 * a_m_s2 * t_s**2

In [11]:
print("С параметрами по умолчанию:", position(4.0))
print("Без ускорения:", position(4.0, a_m_s2=0.0))
print("Другая начальная координата:", position(0.0, x0_m=7.0))

coordinates_m = []
for time_s in times_s:
    coordinates_m.append(position(time_s))
print("Вся серия:", coordinates_m)

С параметрами по умолчанию: 13.0
Без ускорения: 9.0
Другая начальная координата: 7.0
Вся серия: [1.0, 3.25, 6.0, 9.25, 13.0]


## Не «код запустился», а «проверено конкретное свойство»

Контрольные значения получаем **независимо от проверяемого кода**: из формулы, вручную,
из известного предельного случая. Не стоит генерировать «ожидаемый ответ» той же функцией,
которую проверяем.

`assert` вызывает ошибку, если условие ложно. Здесь это короткая учебная проверка;
для проверки входных данных в рабочей функции используем `raise`, не `assert`.

In [12]:
assert isclose(position(0.0, x0_m=7.0), 7.0)
assert isclose(position(4.0, a_m_s2=0.0), 9.0)
assert isclose(position(4.0), 13.0)
assert isclose(position(2.0, x0_m=0.0, v0_m_s=3.0, a_m_s2=-1.0), 4.0)
print("Четыре численные проверки пройдены")

Четыре численные проверки пройдены


## Исключение — сообщение о нарушенном условии

Если результат не имеет смысла для заданных аргументов, лучше сообщить об этом явно,
чем вернуть правдоподобное число. `raise` создает исключение;
`try / except` позволяет обработать **ожидаемый** вид ошибки.

Обычный traceback читаем снизу: тип ошибки и сообщение, затем место вызова.
Не скрываем все ошибки конструкцией `except: pass`.

In [13]:
try:
    position(-1.0)
except ValueError as error:
    print("Ожидаемая ошибка:", type(error).__name__, "—", error)
else:
    raise AssertionError("Функция обязана отклонять отрицательное время")

Ожидаемая ошибка: ValueError — t_s must be non-negative


## Правдоподобный код с неправильной физикой

В следующей функции потерян квадрат времени. Она может выглядеть убедительно,
а два «удобных» теста даже пройдут. Такой код мог написать и человек, и LLM.

In [14]:
def position_with_bug(t_s: float) -> float:
    return 1.0 + 2.0 * t_s + 0.5 * 0.5 * t_s  # Ошибка: нужно t_s**2.


# Ожидаемые значения вычислены вручную, а не ошибочной функцией.
for check_time_s, expected_m in [(0.0, 1.0), (1.0, 3.25), (4.0, 13.0)]:
    actual_m = position_with_bug(check_time_s)
    verdict = "OK" if isclose(actual_m, expected_m) else "ОШИБКА НАЙДЕНА"
    print(f"t={check_time_s:.0f}: получено {actual_m:.2f}; ожидалось {expected_m:.2f} → {verdict}")

t=0: получено 1.00; ожидалось 1.00 → OK
t=1: получено 3.25; ожидалось 3.25 → OK
t=4: получено 10.00; ожидалось 13.00 → ОШИБКА НАЙДЕНА


**Вывод:** выбираем не только удобные примеры, но и случаи, различающие правильную и неправильную модели.
Зеленые тесты — свидетельство проверенных свойств, **не доказательство правильности всего решения**.



<a id="project"></a>
# 4. Файл → модуль → пакет → проект

| Понятие | В нашем примере | Смысл |
|:--|:--|:--|
| Модуль | `model.py` | Файл с моделью и ее функциями |
| Обычный Python-пакет | `course_motion/` с `__init__.py` | Именованная группа модулей |
| Проект | Вся папка рядом с `pyproject.toml` | Код, тесты, инструкции и описание окружения |
| Запуск приложения | `python -m course_motion` | Выполнение `course_motion/__main__.py` |

Для первого занятия используем **обычный пакет** с `__init__.py`. Существуют namespace-пакеты без
этого файла, но сегодня они не нужны. `import` подключает модуль;
**он ничего не устанавливает из интернета**. [S5]

```python
from course_motion.model import position
```

In [15]:
import math
from math import hypot

print("Через имя модуля:", math.sqrt(25.0))
print("Через импортированную функцию:", hypot(3.0, 4.0))

Через имя модуля: 5.0
Через импортированную функцию: 5.0


## Структура нашего учебного проекта

```text
lecture02_project/
├── README.md                      # Что делает проект и как его проверить
├── pyproject.toml                 # Python, зависимости, способ сборки
├── uv.lock                        # Зафиксированное разрешение зависимостей
├── .python-version                # Выбранная версия Python: 3.13
├── .gitignore                     # Что не включать в репозиторий
├── src/
│   └── course_motion/
│       ├── __init__.py
│       ├── model.py               # Только модель
│       └── __main__.py            # Запуск и печать результата
├── tests/
│   └── test_motion.py             # Проверки модели
├── notebooks/
│   └── report_template.ipynb      # Опыт и пояснение, не копия модели
└── .devcontainer/                 # Необязательная среда разработки
```

Папки называются так по **соглашению**, не по магическому правилу Python.
`src` отделяет импортируемый код от остальных файлов. Для штатного импорта пакет
нужно установить в окружение — далее это сделает `uv sync`. [S6]

**Не обязаны создавать десять папок для десяти строк.** Один одноразовый расчет может остаться
в одном файле. Здесь разделение оправдано: модель используют и приложение, и отчет, и тесты.

## Одна модель — несколько способов использования

**`model.py`** содержит уже разобранную функцию `position`. **`__main__.py`**:

```python
from course_motion.model import position


def main() -> None:
    for t_s in range(5):
        print(f"t = {t_s} s; x = {position(t_s):.2f} m")


if __name__ == "__main__":
    main()
```

`range(5)` дает `0, 1, 2, 3, 4`. Блок `if __name__ == "__main__"` отделяет
явный запуск от обычного импорта. Верхнеуровневый код модуля выполняется при первом импорте;
поэтому не прячем там рассылку, установку пакетов или тяжелый расчет. [S5]

**В отчете** достаточно `from course_motion.model import position` и вызова функции.
Не нужно копировать ее реализацию в каждый notebook.

## Живая демонстрация: код работает без памяти notebook

Следующие служебные ячейки создают **новую папку во временном каталоге ОС** и запускают модель
отдельным процессом. Старые файлы не перезаписываются, пакеты не устанавливаются.
Путь выводится ниже; это демонстрационная копия, а не место для постоянного хранения работы.

Чтобы демонстрация работала офлайн даже без `uv`, только здесь запускаем процесс **из папки `src`**:
Python видит пакет в текущей папке. Это не установка пакета. Штатный запуск проекта
**из корня** через `uv` разберем в разделе 6. `sys.path` не изменяем.

Служебная ячейка со строками файлов свернута; ее код можно раскрыть. Основное содержимое файлов
объяснено выше и входит в сопровождающий шаблон проекта.

In [16]:
# Готовые тексты файлов; эта служебная ячейка свернута.
project_files = {'pyproject.toml': '[project]\n'
                   'name = "course-motion"\n'
                   'version = "0.1.0"\n'
                   'description = "Лекция 2: минимальный воспроизводимый Python-проект"\n'
                   'readme = "README.md"\n'
                   'requires-python = ">=3.13,<3.15"\n'
                   'dependencies = []\n'
                   '\n'
                   '[build-system]\n'
                   'requires = ["uv_build==0.10.0"]\n'
                   'build-backend = "uv_build"\n',
 '.python-version': '3.13\n',
 '.gitignore': '.venv/\n__pycache__/\n*.py[cod]\n.ipynb_checkpoints/\n.pytest_cache/\n',
 'src/course_motion/__init__.py': '"""Учебный пакет курса."""\n',
 'src/course_motion/model.py': '"""Одномерное движение тележки с постоянным ускорением."""\n'
                               '\n'
                               '\n'
                               'def position(\n'
                               '    t_s: float,\n'
                               '    x0_m: float = 1.0,\n'
                               '    v0_m_s: float = 2.0,\n'
                               '    a_m_s2: float = 0.5,\n'
                               ') -> float:\n'
                               '    """Вернуть координату в метрах при t_s >= 0. Все аргументы '
                               '— конечные числа."""\n'
                               '    if t_s < 0:\n'
                               '        raise ValueError("t_s must be non-negative")\n'
                               '    return x0_m + v0_m_s * t_s + 0.5 * a_m_s2 * t_s**2\n',
 'src/course_motion/__main__.py': '"""Запуск: uv run python -m course_motion из корня '
                                  'проекта."""\n'
                                  '\n'
                                  'from course_motion.model import position\n'
                                  '\n'
                                  '\n'
                                  'def main() -> None:\n'
                                  '    for t_s in range(5):\n'
                                  '        x_m = position(t_s)\n'
                                  '        print(f"t = {t_s:2} s; x = {x_m:5.2f} m")\n'
                                  '\n'
                                  '\n'
                                  'if __name__ == "__main__":\n'
                                  '    main()\n',
 'tests/test_motion.py': '"""Проверки контракта модели; достаточно стандартной библиотеки '
                         'Python."""\n'
                         '\n'
                         'import unittest\n'
                         '\n'
                         'from course_motion.model import position\n'
                         '\n'
                         '\n'
                         'class TestPosition(unittest.TestCase):\n'
                         '    def test_initial_position(self):\n'
                         '        self.assertAlmostEqual(position(0.0, x0_m=7.0), 7.0)\n'
                         '\n'
                         '    def test_uniform_motion(self):\n'
                         '        self.assertAlmostEqual(position(4.0, x0_m=1.0, v0_m_s=2.0, '
                         'a_m_s2=0.0), 9.0)\n'
                         '\n'
                         '    def test_accelerated_motion(self):\n'
                         '        # Независимый ручной расчет: 1 + 2*4 + 0.5*0.5*4**2 = 13.\n'
                         '        self.assertAlmostEqual(position(4.0), 13.0)\n'
                         '\n'
                         '    def test_negative_acceleration(self):\n'
                         '        self.assertAlmostEqual(position(2.0, x0_m=0.0, v0_m_s=3.0, '
                         'a_m_s2=-1.0), 4.0)\n'
                         '\n'
                         '    def test_negative_time_is_rejected(self):\n'
                         '        with self.assertRaisesRegex(ValueError, "non-negative"):\n'
                         '            position(-0.1)\n'
                         '\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    unittest.main()\n',
 'README.md': '# Демонстрация лекции 2\n'
              '\n'
              'Запуск из корня после `uv sync`: `uv run python -m course_motion`.\n',
 'uv.lock': 'version = 1\n'
            'revision = 3\n'
            'requires-python = ">=3.13, <3.15"\n'
            '\n'
            '[[package]]\n'
            'name = "course-motion"\n'
            'version = "0.1.0"\n'
            'source = { editable = "." }\n'}

In [17]:
import tempfile

# Каждый запуск создает отдельную папку; существующие проекты не изменяются.
demo_root = Path(tempfile.mkdtemp(prefix="lecture02_"))
for relative_name, content in project_files.items():
    target = demo_root / relative_name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

print("Демонстрационный проект:", demo_root)
for file in sorted(demo_root.rglob("*")):
    if file.is_file():
        print("  ", file.relative_to(demo_root))

Демонстрационный проект: /tmp/lecture02_hoy6ovj_
   .gitignore
   .python-version
   README.md
   pyproject.toml
   src/course_motion/__init__.py
   src/course_motion/__main__.py
   src/course_motion/model.py
   tests/test_motion.py
   uv.lock


In [18]:
import subprocess

run_result = subprocess.run(
    [sys.executable, "-X", "utf8", "-m", "course_motion"],
    cwd=demo_root / "src",
    capture_output=True, text=True, encoding="utf-8", check=True, timeout=15,
)
print(run_result.stdout)

t =  0 s; x =  1.00 m
t =  1 s; x =  3.25 m
t =  2 s; x =  6.00 m
t =  3 s; x =  9.25 m
t =  4 s; x = 13.00 m



## Проверка в отдельном процессе

В шаблоне — **пять автоматических тестов**: начальная координата, нулевое ускорение,
ускоренное движение, отрицательное ускорение и отказ при отрицательном времени.
Ожидаемые числа заданы независимо от реализации.

Используем `unittest` из стандартной библиотеки: на демонстрации не нужна установка `pytest`.
Форма тестового класса — готовый шаблон; **ООП не является темой этой лекции**.
Позже эти же тесты можно запустить командой `uv run pytest` после добавления pytest. [S7]

In [19]:
test_result = subprocess.run(
    [sys.executable, "-X", "utf8", "-m", "unittest", "discover", "-s", str(demo_root / "tests"), "-v"],
    cwd=demo_root / "src",
    capture_output=True, text=True, encoding="utf-8", check=True, timeout=15,
)
print(test_result.stdout + test_result.stderr)

test_accelerated_motion (test_motion.TestPosition.test_accelerated_motion) ... ok
test_initial_position (test_motion.TestPosition.test_initial_position) ... ok
test_negative_acceleration (test_motion.TestPosition.test_negative_acceleration) ... ok
test_negative_time_is_rejected (test_motion.TestPosition.test_negative_time_is_rejected) ... ok
test_uniform_motion (test_motion.TestPosition.test_uniform_motion) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.000s

OK



## Notebook и проект не конкурируют

| В файлах `.py` | В notebook |
|:--|:--|
| Вычислительная модель и повторно используемые функции | Постановка конкретного опыта |
| Проверки контракта в `tests/` | Выбор параметров и анализ результатов |
| Управляемая точка запуска | Таблицы, графики, формулы и выводы |

Короткий исследовательский прототип в notebook нормален. Переносим код в модуль,
когда он повторяется, нужен нескольким опытам или требует отдельной проверки.

**Важная особенность:** после изменения `.py` уже импортированный модуль может остаться
в памяти kernel. Для надежной проверки обновления — **Restart Kernel → Run All**.
Editable-установка пакета не перезагружает уже импортированный Python-код. [S5, S8]

<details>
<summary><b>Заметка преподавателю</b></summary>

Переход к старым лабораторным: вычислительная часть — отдельный модуль, notebook — постановка опыта и оформление. Не переносите сейчас в лекцию оптимизацию, ОДУ и полный scientific stack.

</details>

<a id="venv"></a>
# 5. Окружение: четыре разных вопроса

| Вопрос | Ответственный механизм |
|:--|:--|
| Какая программа исполняет код? | Интерпретатор Python |
| Какие Python-пакеты доступны этому проекту? | Виртуальное окружение, обычно `.venv` |
| Кто устанавливает пакеты? | `pip` или `uv` |
| Кто описывает ОС и системные инструменты? | Например, Dev Container |

**`pip` не создает изоляцию сам по себе.** В базовом пути `venv` создает окружение,
а `pip` устанавливает туда пакеты. `python -m pip` явно связывает установщик
с выбранным интерпретатором. [S9, S10]

```text
Проект A → .venv A → собственный набор установленных Python-пакетов
Проект B → .venv B → другой набор установленных Python-пакетов
```

Окружение не изолирует пользовательские файлы, сеть и права доступа процесса.
**`.venv` — не безопасная песочница для чужого кода.**

## Базовый путь: `venv + pip` — понимать, но не заучивать

**Команды ниже — терминал, не Python-ячейка.** Выполнять в отдельной учебной папке,
а не поверх `.venv`, которой уже управляет `uv`.

**Windows · PowerShell**

```powershell
py -3.13 -m venv .venv
.\.venv\Scripts\python.exe -m pip install numpy
.\.venv\Scripts\python.exe -c "import numpy; print(numpy.__version__)"
```

**Linux / macOS · терминал с установленным Python 3.13**

```bash
python3.13 -m venv .venv
./.venv/bin/python -m pip install numpy
./.venv/bin/python -c "import numpy; print(numpy.__version__)"
```

Это демонстрация механизма установки, не зависимость нашего расчета.
Активация не обязательна: мы явно указали интерпретатор. Если пакет уже установлен
в один Python, другой kernel от этого не изменится. [S9, S10]

Если команды `py -3.13` или `python3.13` нет, соответствующий интерпретатор еще не установлен
или не найден оболочкой. Не исправляем это случайной установкой библиотек.

## Типичная ошибка: «Я уже установил, почему `import` не работает?»

Сначала смотрим на факты, не переустанавливаем все подряд:

```python
import sys
print(sys.executable)
```

Сравниваем путь ядра с интерпретатором, которым устанавливали пакет. В VS Code выбор Python
для обычных файлов и выбор **kernel конкретного notebook** — отдельные действия. [S2]

`!pip install ...` зависит от найденного в оболочке `pip`. IPython-команда `%pip`
ориентирована на окружение текущего kernel, **но не ведет `pyproject.toml` и `uv.lock`**;
кроме того, pip может отсутствовать в uv-окружении. В нашем рабочем процессе добавляем
зависимости через **`uv add` в терминале проекта**, затем при необходимости перезапускаем kernel. [S1]

<a id="uv"></a>
# 6. uv: описание проекта → окружение → запуск

Сегодня основной путь курса — `uv`. Не надо одновременно осваивать несколько менеджеров.
`pip + venv` оставляем как базовую модель и способ читать чужие инструкции.

| Объект | Что означает | Сохраняем вместе с проектом? |
|:--|:--|:--|
| `pyproject.toml` | Требования и настройки проекта | Да |
| `uv.lock` | Выбранные точные версии зависимостей, включая транзитивные | Да |
| `.python-version` | Какой Python выбираем для работы | Да |
| `.venv/` | Установленное локальное окружение | Нет |

`uv run` по умолчанию проверяет lock-файл и окружение перед выполнением команды.
`uv sync` приводит окружение в соответствие с описанием; это не синоним
«обновить все пакеты до самых новых версий». [S8, S11]

**Файлы окружения восстанавливаем по описанию, а не пересылаем папку `.venv`.**

## `pyproject.toml` на конкретном примере

```toml
[project]
name = "course-motion"
version = "0.1.0"
description = "Лекция 2: минимальный воспроизводимый Python-проект"
readme = "README.md"
requires-python = ">=3.13,<3.15"
dependencies = []

[build-system]
requires = ["uv_build==0.10.0"]
build-backend = "uv_build"

```

`[project]` описывает проект. `requires-python` — **диапазон совместимости**, а не команда
установки Python. `dependencies = []` честно означает: нашей модели не нужны сторонние
библиотеки. `[build-system]` позволяет установить код из `src/` как пакет;
публиковать его на PyPI для этого не требуется. [S6, S12]

В примере закреплена версия сборочного backend, проверенная при подготовке.
Внутреннее устройство сборки сегодня не изучаем. `course-motion` — имя дистрибутива,
а `course_motion` — имя импортируемого пакета: дефис нельзя использовать в `import`.

**Версия курса:** `.python-version` содержит `3.13`. Это базовая линия данного материала,
не заявление о самой новой версии Python. Patch-релиз этим файлом не закреплен.

In [20]:
import tomllib

project_description = tomllib.loads(project_files["pyproject.toml"])
print("Проект:", project_description["project"]["name"])
print("Поддерживаемый Python:", project_description["project"]["requires-python"])
print("Зависимости модели:", project_description["project"]["dependencies"])

Проект: course-motion
Поддерживаемый Python: >=3.13,<3.15
Зависимости модели: []


## Два сценария: получить проект и начать новый

### A. Уже есть проект преподавателя

Распакуйте сопровождающий шаблон и откройте терминал **в его корне**, рядом с `pyproject.toml`:

```bash
uv sync --locked
uv run --locked python -m course_motion
uv run --locked python -m unittest discover -s tests -v
```

`--locked` запрещает незаметно переписать lock-файл: если он не согласован с проектом,
будет ошибка. Флаг `--frozen` означает другое — не проверять актуальность lock;
для первичной проверки чужой работы его не используем. [S8]

### B. Создаем свой проект

```bash
uv init --package --python 3.13 my-lab
cd my-lab
uv sync
```

Затем создаем модель, точку запуска и тесты по нашему примеру.
Явный `--package` показывает намерение создать устанавливаемый пакет и не полагается
на различия поведения `uv init` между версиями. Генератор создает каркас,
**не готовое решение инженерной задачи**. [S13]

В Run All эти команды **не выполняются**. Первое скачивание Python, backend или библиотек
может потребовать интернет; настройку преподаватель проверяет до лекции.

## Добавляем библиотеку, только когда она понадобилась

**Терминал · в корне проекта**

```bash
uv add numpy
uv add --dev ipykernel pytest
uv run python -c "import numpy; print(numpy.__version__)"
uv run pytest -q
```

Первая команда добавляет зависимость приложения; вторая — инструменты разработки.
Группа `dev` по умолчанию включается в `uv sync` и `uv run`. `uv add` меняет
и описание проекта, и lock-файл, и окружение. [S11, S14]

В этой лекции NumPy **не нужен** для модели; команды — пример следующего шага,
а не обязательная установка «на всякий случай». `pytest` может запускать наши готовые
тесты на unittest. Исходный шаблон работает без обеих библиотек. [S7]

```bash
uv remove numpy
uv tree
```

Удаление зависимости и просмотр дерева — действия с проектом. Установка через
`uv pip install ...` не равна `uv add`: сама по себе она не сохраняет требование
в `pyproject.toml` и `uv.lock`. [S1, S11]

## Lock-файл: чего он гарантировать не может

В минимальном шаблоне сторонних зависимостей нет, поэтому исходный `uv.lock` короткий.
После `uv add` в нем появятся реальные выбранные версии и зависимости зависимостей.
**Не пишем эти записи вручную.**

```text
pyproject.toml  →  uv разрешает зависимости  →  uv.lock
                                               ↓
                                            uv sync
                                               ↓
                                        локальная .venv
```

Lock-файл не содержит самих библиотек и не фиксирует входные данные, ОС, GPU, случайные числа
или поведение внешнего сервиса. Совпадение зависимостей — часть воспроизводимости,
а не обещание побитово одинакового результата на любом компьютере.

При целевом обновлении сначала меняем зависимости, затем проверяем тесты и сохраняем
обновленный lock. Не выполняем массовый upgrade перед сдачей работы без необходимости. [S8]

## Связываем Jupyter с uv-проектом

**Терминал · в корне проекта**

```bash
uv add --dev ipykernel
```

В VS Code откройте **всю папку проекта**, затем нужный `.ipynb`.
В notebook: **Select Kernel → Python Environments → окружение проекта**. [S1, S2]

| ОС | Ожидаемый интерпретатор |
|:--|:--|
| Windows | `<проект>\.venv\Scripts\python.exe` |
| Linux / macOS | `<проект>/.venv/bin/python` |

В первой ячейке проверьте `sys.executable`. Не выбирайте kernel только по похожему названию
«Python 3»: на компьютере их может быть несколько. После смены окружения — Restart Kernel → Run All.

**Проверка в отчете после установки пакета:**

```python
from course_motion.model import position
print(position(4.0))  # 13.0
```

Для этого готов `notebooks/report_template.ipynb`. Он намеренно импортирует пакет,
а не дублирует функцию. Создание виртуального окружения в терминале само по себе
не переключает уже работающий notebook.

<a id="container"></a>
# 7. Dev Container: следующий уровень, не замена uv

| `.venv` | Dev Container |
|:--|:--|
| Разделяет наборы Python-пакетов | Описывает контейнерную среду разработки |
| Не устанавливает ОС и системные утилиты | Может включать Python, системные библиотеки и утилиты |
| Обычно достаточно для нашей первой модели | Полезен, когда важны одинаковые системные зависимости |

```text
Dev Container: ОС / системные инструменты / Python / uv
└── окружение Python этого проекта
    └── установленный пакет course_motion
        └── приложение, тесты, notebook
```

`devcontainer.json` описывает среду для поддерживающего ее инструмента;
это **не программа** и не самостоятельный контейнерный движок. Для пути VS Code + Docker
нужны работающий Docker и расширение Dev Containers. [S15, S16]

На этом занятии: понять назначение и посмотреть готовую конфигурацию.
Массовую настройку Docker/WSL во время лекции не проводим.

## Минимальная конфигурация для нашего проекта

**`.devcontainer/Dockerfile`**

```dockerfile
FROM ghcr.io/astral-sh/uv:0.10.0 AS uv_binary
FROM mcr.microsoft.com/devcontainers/python:1-3.13-bookworm
COPY --from=uv_binary /uv /uvx /usr/local/bin/

```

**Главные поля `.devcontainer/devcontainer.json`**

```json
{
  "name": "Lecture 02 — Python project",
  "build": {"dockerfile": "Dockerfile"},
  "remoteUser": "vscode",
  "containerEnv": {
    "UV_PROJECT_ENVIRONMENT": "/home/vscode/.venvs/course-motion"
  },
  "postCreateCommand": "uv sync --locked"
}
```

В полном файле шаблона также указаны расширения Python/Jupyter и путь интерпретатора VS Code.
Для notebook сначала добавьте `ipykernel` в dev-зависимости, чтобы `uv sync --locked` установил его
внутри контейнера. Затем выберите контейнерный kernel. [S1, S15]

**Почему окружение не в общей `.venv`?** Код примонтирован с хоста, но окружения Windows и Linux
не взаимозаменяемы. Переменная `UV_PROJECT_ENVIRONMENT` задает отдельный путь внутри контейнера. [S6, S9]

Это обзорный шаблон: конфигурация статически проверена, **сборка Docker здесь не проверялась**.
Образы потребуется скачать; теги без digest не гарантируют неизменность образов.

## Изоляция окружения ≠ разрешение запускать что угодно

Контейнер может менять файлы примонтированного проекта и обращаться в сеть.
Внимательно смотрим на команды установки, монтирования, доступ к секретам и Docker socket.
**Ни `.venv`, ни сам факт Dev Container не делают неизвестный код LLM безопасным.**

Практическое правило для курса: не передавать агенту пароли, почтовые токены,
персональные данные студентов и доступы, которые не нужны для задания.
При открытии чужого проекта сначала прочитать README, настройки контейнера и команды запуска.

Контейнер нужен из-за конкретного требования к среде, а не для добавления еще одной технологии.

<a id="llm"></a>
# 8. Как поручить изменение LLM и остаться автором решения

Вместо «улучши этот код» задаем границы и критерии приемки:

```text
Прочитай README.md, pyproject.toml, src/course_motion/model.py
и tests/test_motion.py. Кратко объясни структуру и текущий контракт.

Добавь в model.py функцию velocity(t_s, v0_m_s=2.0, a_m_s2=0.5),
возвращающую скорость в м/с: v = v0 + a*t.
Аргументы — конечные числовые значения; отрицательное время отклоняется ValueError.
Не меняй существующую position и ее значения по умолчанию.

Добавь тесты: t=0; a=0; t=4 при параметрах по умолчанию;
отрицательное ускорение; отрицательное время.
Не добавляй зависимости и не удаляй существующие тесты.
Запусти: uv run python -m unittest discover -s tests -v.
Покажи, какие файлы изменены и что реально проверено.
```

LLM здесь помогает написать изменение. **Студент отвечает за смысл формулы,
единицы, разумность тестов и воспроизводимый запуск.** Ту же задачу можно выполнить вручную:
платный доступ к LLM не требуется.

## Выходной контроль · последние 3 минуты

**1.** Одногруппник прислал `src/`, `pyproject.toml` и `uv.lock`, но не `.venv`. Это ошибка?

**2.** `uv run python -c "import numpy"` работает, а notebook — нет. Что проверяем первым?

**3.** Код LLM проходит тесты при `t=0` и `t=1`. Достаточно ли этого, чтобы принять формулу движения?

<details>
<summary>Ориентиры ответа</summary>

1. Отсутствие `.venv` нормально: она восстанавливается через `uv sync --locked`.
Нужны также необходимые исходные данные, инструкция и прочие файлы, от которых зависит работа.
2. `sys.executable` в notebook и выбранный kernel: терминал и kernel могут использовать разные окружения.
3. Нет. Эти точки не различают `t` и `t**2`; нужны независимые контрольные случаи и проверка смысла.

</details>

---

### Что должно остаться после лекции

**Понимаю код → нахожу его в проекте → выбираю окружение → запускаю → проверяю.**

Уметь объяснить этот маршрут важнее, чем помнить все команды и синтаксические конструкции.
Основной маршрут на 90 минут **закончен**. Далее — приложения для самостоятельной работы.

<a id="setup"></a>
# Приложение A. Подготовка до занятия

**Преподавателю:** открыть этот notebook в рабочем Jupyter, проверить Run All и сохранить выводы.
Заранее выполнить команды `uv` на отдельной копии шаблона. Docker показывать только после
успешного пробного запуска на своей машине; не делать его обязательным условием для аудитории.

**Студенту:** основной путь — VS Code с расширениями Python и Jupyter, `uv`, Python 3.13.
Это рабочее предположение для данного материала; у кого уже настроен JupyterLab,
можно использовать его без смены редактора.

**Установка uv — Windows с WinGet:**

```powershell
winget install --id=astral-sh.uv -e
```

**macOS с Homebrew:**

```bash
brew install uv
```

Для Linux и других вариантов используйте официальную инструкцию Astral [S17].
Она предлагает, в частности, установочный скрипт; перед запуском проверьте источник
и содержимое. Не выполняйте неизвестные shell-команды только потому, что их предложила модель.
После установки откройте новый терминал и проверьте:

```bash
uv --version
uv python install 3.13
```

Последняя команда скачивает Python, если нужный управляемый интерпретатор еще не установлен.
Для первой установки нужен доступ к соответствующим серверам. Это не обещание доступности
сети из любой аудитории: подготовьте среду и резервный локальный показ заранее. [S18]

## Первый запуск готового комплекта

Распакуйте `lecture02_project.zip`. В терминале откройте папку, где лежит `pyproject.toml`:

```bash
uv sync --locked
uv run --locked python -m course_motion
uv run --locked python -m unittest discover -s tests -v
uv add --dev ipykernel
```

Откройте `notebooks/lecture02.ipynb`, выберите kernel `.venv` и выполните Run All.
Первые три команды проверяют исходный шаблон; последняя осознанно добавляет инструмент разработки
и меняет описание проекта/lock. Сохраните оба обновленных файла.

**Без VS Code — в браузере:**

```bash
uv add --dev jupyterlab
uv run jupyter lab
```

Откройте notebook в появившемся JupyterLab; проверьте путь ядра первой ячейкой.
`uv tool run jupyter lab` — иной режим: инструмент может работать в отдельном окружении.
Для первого занятия не смешиваем эти способы. [S1]

Полный scientific stack и Docker для выполнения этого notebook не нужны.

## Диагностика: симптом → первая проверка

| Симптом | Что проверить / сделать |
|:--|:--|
| `uv` не найден | Установку и PATH; открыть новый терминал |
| Пакет виден в терминале, но не notebook | `sys.executable` и kernel notebook |
| `No module named course_motion` | Находитесь ли в корне проекта; выполнен ли `uv sync`; какой kernel выбран |
| `No module named pip` в uv-окружении | Для зависимостей проекта используйте `uv add`; pip не обязан быть установлен |
| Не выполняется `Activate.ps1` | Активация не нужна для `uv run`; не меняйте политику безопасности ради этого |
| После изменения `.py` старый результат | Restart Kernel → Run All; проверьте, какой файл импортирован |
| Lock-файл устарел | Сверить изменения `pyproject.toml`; обновить lock осознанно, не скрывать проблему `--frozen` |
| Импорт ломается только из-за папки `notebooks/` | Установить пакет, не добавлять абсолютный путь в `sys.path` |
| Docker не запускается | Вернуться к локальному uv-пути; проблема Docker не равна ошибке модели |

Для проверки импортированного файла:

```python
import course_motion.model
print(course_motion.model.__file__)
```

Не называйте свои файлы `math.py`, `json.py`, `numpy.py`, `typing.py`:
они могут перекрывать ожидаемые модули. Не присваивайте собственные данные именам
`list`, `dict`, `str`, если дальше нужны одноименные встроенные типы. [S5]

<a id="cheatsheet"></a>
# Приложение B. Короткая шпаргалка Python

Это необязательное продолжение. Примеры сохранены или переработаны из старых двух семинаров.

## Еще один старый пример: расстояние между точками

В старом введении уже была удачная функция `distance(x0, y0, x1, y1)` с docstring.
Сохраняем задачу; добавляем единицы, аннотации и контрольную тройку **3–4–5**.

In [21]:
def distance(x0: float, y0: float, x1: float, y1: float) -> float:
    """Расстояние между точками; все координаты заданы в одних единицах."""
    return hypot(x1 - x0, y1 - y0)


assert isclose(distance(0.0, 0.0, 3.0, 4.0), 5.0)
print("Расстояние:", distance(0.0, 0.0, 3.0, 4.0))
print(distance.__doc__)

Расстояние: 5.0
Расстояние между точками; все координаты заданы в одних единицах.


## Копирование, цикл и компактная запись

`b = a` **не копирует список**. Оба имени могут указывать на один изменяемый объект.
`copy()` создает поверхностную копию; для списка из чисел этого достаточно.
У вложенных коллекций есть дополнительные нюансы — здесь их не разбираем.

In [22]:
original = [1.0, 2.0]
alias = original
independent = original.copy()
alias.append(3.0)
print("Исходный список:", original)
print("Независимая копия:", independent)

# Comprehension — компактный вариант уже понятного цикла.
compact_coordinates = [position(time_s) for time_s in times_s]
assert compact_coordinates == coordinates_m
print(compact_coordinates)

for index, value in enumerate(compact_coordinates):
    print(f"Точка {index}: {value:.2f} м")

Исходный список: [1.0, 2.0, 3.0]
Независимая копия: [1.0, 2.0]
[1.0, 3.25, 6.0, 9.25, 13.0]
Точка 0: 1.00 м
Точка 1: 3.25 м
Точка 2: 6.00 м
Точка 3: 9.25 м
Точка 4: 13.00 м


## `while`, `break`, `continue` — узнавать в чужом коде

`while` повторяет блок, пока условие истинно. Скорость программы зависит от выполняемой работы
и реализации, а не от тезиса «while универсальный, поэтому медленный».
Изменяйте условие завершения, иначе цикл может не закончиться.

In [23]:
remaining = 3
while remaining > 0:
    print("Осталось шагов:", remaining)
    remaining -= 1

for sample in [-1.0, 0.0, 1.0, 2.0, 3.0]:
    if sample < 0:
        continue  # Перейти к следующему элементу.
    if sample > 2:
        break     # Завершить цикл.
    print("Принято время:", sample)

Осталось шагов: 3
Осталось шагов: 2
Осталось шагов: 1
Принято время: 0.0
Принято время: 1.0
Принято время: 2.0


## Аннотация типа сама не запрещает неподходящий аргумент

Следующий пример **специально нарушает** объявленный контракт. Python не останавливается
на аннотации; операция умножения определена и для строки. Это демонстрация,
не рекомендуемый способ использования функции. [S4]

In [24]:
def double(value: float) -> float:
    return value * 2


print("Число:", double(3.0))
print("Нарушение контракта:", double("3"))

Число: 6.0
Нарушение контракта: 33


## Файл с результатами: путь и явная кодировка

Переработка материала о файлах: используем `pathlib`, а не абсолютные пути
в личной папке преподавателя. Файл ниже записывается только в созданную демонстрационную папку.

`JSON` удобен для простых данных, но сохраняет не любые объекты Python.
Не загружайте неизвестный `pickle`: это не безопасный формат обмена чужими данными. [S19]

In [25]:
import json

result_data = {"times_s": times_s, "positions_m": coordinates_m}
result_path = demo_root / "result.json"
result_path.write_text(json.dumps(result_data, ensure_ascii=False, indent=2), encoding="utf-8")
loaded_data = json.loads(result_path.read_text(encoding="utf-8"))
assert loaded_data == result_data
print(result_path.read_text(encoding="utf-8"))

{
  "times_s": [
    0.0,
    1.0,
    2.0,
    3.0,
    4.0
  ],
  "positions_m": [
    1.0,
    3.25,
    6.0,
    9.25,
    13.0
  ]
}


<a id="practice"></a>
# Приложение C. Небольшая самостоятельная практика

**Результат:** дополнить готовый проект и сдать проверяемое изменение, а не текст переписки с моделью.
Ориентир — отдельная короткая практика после лекции; она не включена в 90 минут.

### Задание

Добавьте `velocity` по постановке раздела 8. В `report_template.ipynb` запишите формулу,
выведите скорости в те же моменты времени и объясните один случай отрицательного ускорения.
Добавьте пять тестов; существующие тесты должны продолжать проходить.

LLM разрешен, но не обязателен. Перед принятием предложенного кода прочитайте diff
и самостоятельно проверьте одно ожидаемое значение. Не добавляйте библиотеки, если хватает Python.

### Критерии приемки

| Критерий | Наблюдаемая проверка |
|:--|:--|
| Функция соответствует контракту | Ручной расчет и пять новых тестов |
| Старое поведение не сломано | Старые пять тестов по-прежнему проходят |
| Код находится в модели | Нет копии функции в notebook |
| Работу можно воспроизвести | Команды из README, согласованные pyproject/lock |
| Автор понимает изменение | Объяснение единиц, формулы и граничного случая |

После изменений выполните из корня:

```bash
uv run --locked python -m unittest discover -s tests -v
```

Затем Restart Kernel → Run All в отчете. Передайте исходники, тесты, README,
`pyproject.toml`, `uv.lock`, `.python-version` и notebook. `.venv` и секреты не прикладывайте.

## Карточка изменения для обратной связи

В конец своего отчета добавьте **три коротких абзаца**, а не полный лог диалога:

**Постановка.** Что нужно было изменить и какие ограничения сохранить?

**Проверка.** Какими командами проверено? Какое контрольное значение вычислено самостоятельно?

**Ответственность.** Что предложила LLM, что вы исправили и что пока не проверено?
Если работали без LLM, так и укажите.

Преподавателю достаточно предложить небольшое изменение входных данных и попросить
предсказать результат до запуска. Это проверяет понимание конкретной работы,
а не способность красиво оформить сгенерированный ответ.

<a id="sources"></a>
# Приложение D. Что переработано из существующих материалов

Это не механическое объединение старых файлов. Основной маршрут сокращен до одного занятия,
а новые блоки связывают язык с воспроизводимым проектом.

| Материал в предоставленных архивах | Что использовано | Что изменено |
|:--|:--|:--|
| `old(1).zip → неделя 1-2/семинар 1/Введение в Python.ipynb` — 206 ячеек | Kernel и порядок вычислений; типы; `0.1 + 0.2`; условия; функции; списки; кортежи; `distance` | Убраны незавершенные/падающие примеры из Run All; добавлены единицы, type hints, контрольные значения |
| `old(1).zip → неделя 1-2/семинар 2/семинар 2.ipynb` — 236 ячеек | Словари, цикл, модули, импорт, установка пакетов | Классы/наследование и NumPy отложены; абсолютный `sys.path.append` заменен установкой пакета; добавлены uv и kernel selection |
| `old(1).zip → неделя 9-10/работа с файлами/Работа с файлами.ipynb` | Идея сохранения результатов и JSON | Короткий самостоятельный пример с pathlib и UTF-8, без личных абсолютных путей |
| `old_labs.zip` — преподавательские задания по графикам и интегрированию | Формат вычислительного опыта: постановка → расчет → результат | Подготовлен переход к отдельной модели и notebook-отчету; сами сложные задачи сюда не перенесены |
| `homework.zip → дз2025.ipynb` | Организационный принцип: готовая библиотека и отдельный документ с использованием | Сохранено разделение модели и отчета; предметный расчет не переносится во вводную лекцию по Python |

В двух старых семинарах суммарно **442 ячейки**. Индивидуальные студенческие решения,
персональные данные и чужие выходные файлы в новый материал не включены.

**Содержательные исправления:** `None` — значение; ключи словаря должны быть хешируемыми,
а не просто «любыми неизменяемыми»; универсальность `while` не объясняет его скорость;
`pip` не является механизмом изоляции; type hints не обеспечивают runtime-валидацию.

## Официальная документация

Ссылки проверены **14 сентября 2026 года**. Примеры и педагогический маршрут адаптированы
для этого курса; конкретные технические свойства можно уточнить в первоисточниках.

- **[S1]** [Astral — Using uv with Jupyter](https://docs.astral.sh/uv/guides/integration/jupyter/)
- **[S2]** [VS Code — Manage Jupyter Kernels](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)
- **[S3]** [Python — Floating-Point Arithmetic](https://docs.python.org/3/tutorial/floatingpoint.html)
- **[S4]** [Python — typing: Support for type hints](https://docs.python.org/3/library/typing.html)
- **[S5]** [Python — Modules and Packages](https://docs.python.org/3/tutorial/modules.html)
- **[S6]** [Astral — Configuring projects](https://docs.astral.sh/uv/concepts/projects/config/)
- **[S7]** [pytest — How to use unittest-based tests](https://docs.pytest.org/en/stable/how-to/unittest.html)
- **[S8]** [Astral — Locking and syncing](https://docs.astral.sh/uv/concepts/projects/sync/)
- **[S9]** [Python — venv](https://docs.python.org/3/library/venv.html)
- **[S10]** [PyPA — Install packages in a virtual environment using pip and venv](https://packaging.python.org/en/latest/guides/installing-using-pip-and-virtual-environments/)
- **[S11]** [Astral — Working on projects](https://docs.astral.sh/uv/guides/projects/)
- **[S12]** [PyPA — Writing your pyproject.toml](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/)
- **[S13]** [Astral — Creating projects](https://docs.astral.sh/uv/concepts/projects/init/)
- **[S14]** [Astral — Managing dependencies](https://docs.astral.sh/uv/concepts/projects/dependencies/)
- **[S15]** [VS Code — Dev Container metadata reference](https://code.visualstudio.com/docs/devcontainers/devcontainerjson-reference)
- **[S16]** [VS Code — Developing inside a Container](https://code.visualstudio.com/docs/devcontainers/containers)
- **[S17]** [Astral — Installing uv](https://docs.astral.sh/uv/getting-started/installation/)
- **[S18]** [Astral — Installing Python](https://docs.astral.sh/uv/guides/install-python/)
- **[S19]** [Python — pickle: предупреждение о небезопасной десериализации](https://docs.python.org/3/library/pickle.html)

## Проверка этого экземпляра

**Среда подготовки:** Linux, Python 3.13.5, uv 0.10.0.
Все 25 ячеек кода выполнены с чистым ядром; повторный полный проход также проверен.
Пример проекта установлен через `uv sync --locked`; пять тестов проходят.
Намеренная замена `t_s**2` на `t_s` в модели обнаруживается двумя тестами.

Пути и выводы в сохраненных ячейках относятся к среде подготовки; у вас они будут другими.
Ни одна исполняемая ячейка не устанавливает пакеты и не требует доступа к сети.
**Не проверялись фактические установки на Windows/macOS и сборка Dev Container.**
Инструкции сверены с официальной документацией; проверка конфигурации контейнера — статическая.

Подробный протокол находится в `VALIDATION.md` сопровождающего архива.